[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drksci/phosphene/blob/main/notebooks/04_visualize_results.ipynb)

# 04 · Visualise results

Companion to 03: metrics, error gallery, and the end-to-end **terminal → A2UI** stream —
how many bytes the incremental A2UI stream needs per keyframe vs. the raw VT stream and a naive
full re-render, plus an interactive replay with **native** and **faithful** (style-layer) renders.

In [ ]:
#@title Setup: clone + install (re-run safe)
import os, sys
if not os.path.exists("/content/phosphene"):
    !git clone -q https://github.com/drksci/phosphene /content/phosphene
%cd /content/phosphene
!git pull -q
!pip install -q -e packages/vtcore -e packages/vttui -e "packages/vtm[train,data,label]"

#@markdown Persist data/models to Google Drive (survives runtime resets) or keep them in /content.
USE_DRIVE = False  #@param {type:"boolean"}
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = "/content/drive/MyDrive/phosphene"
else:
    DATA = "/content/phosphene_data"
os.makedirs(DATA, exist_ok=True)
print("DATA =", DATA)

In [ ]:
import json, glob, numpy as np, matplotlib.pyplot as plt
from vtcore import load_frames
from vtm.train import load, predict
from vtm.labeling.heuristics import label
from vtm.metrics import confusion, report
from vtm.viz import plot_iou, plot_confusion, compare_html, show, replay
model = load(f"{DATA}/runs/vtm/vtm.pt", device="cuda" if __import__("torch").cuda.is_available() else "cpu")
hist = json.load(open(f"{DATA}/runs/vtm/history.json"))
split = "gold_test" if os.path.exists(f"{DATA}/gold_test.npz") else "synth_val"
fr, ex = load_frames(f"{DATA}/{split}.npz"); gold = ex["roles"]
pm = [predict(model, f)[0] for f in fr]; ph = [label(f)[0] for f in fr]
rm, rh = report(confusion(pm, gold)), report(confusion(ph, gold))
print(split, "model mIoU", round(rm["macro_iou"], 3), "| heuristics", round(rh["macro_iou"], 3))

## Training curve + per-class IoU

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 3.5), gridspec_kw={"width_ratios": [1, 3]})
ax[0].plot([h["loss"] for h in hist], label="train loss")
if "macro_iou" in hist[0]: ax[0].plot([h["macro_iou"] for h in hist], label="val mIoU")
ax[0].legend(); ax[0].set_xlabel("epoch")
plot_iou({"VTM": rm, "heuristics": rh}, ax=ax[1]); plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
plot_confusion(confusion(pm, gold), ax[0], "VTM"); plot_confusion(confusion(ph, gold), ax[1], "heuristics"); plt.tight_layout()

## Error gallery — worst frames for the model

In [ ]:
err = [float(((p != g) & ((p > 0) | (g > 0))).sum() / max(((p > 0) | (g > 0)).sum(), 1)) for p, g in zip(pm, gold)]
for i in np.argsort(err)[::-1][:4]:
    print(f"frame {i}  error {err[i]:.2f}")
    show(compare_html(fr[i], {"gold": gold[i], "VTM": pm[i], "heuristics": ph[i]}))

## Terminal → A2UI stream
Pick a real recording (or a synthetic one) and stream it through VTM → compiler → reconciler.

In [ ]:
from vtcore import load_cast
from vtm.synth import synth_cast
from vtm.stream import cast_to_a2ui
from vtcore.emulator import keyframes
paths = sorted(glob.glob("/content/recordings/**/record.*", recursive=True))
cast = load_cast(paths[7]) if paths else synth_cast(42)
labeler = lambda f: predict(model, f)[0]
msgs, stats = cast_to_a2ui(cast, labeler, max_frames=120)
kf = list(keyframes(cast, max_frames=120))
vt, inc, full = (np.array([s[k] for s in stats]) for k in ("vt_bytes", "a2ui_bytes", "full_bytes"))
fig, ax = plt.subplots(1, 2, figsize=(16, 3.5))
ax[0].plot(full, label="A2UI full re-render"); ax[0].plot(inc, label="A2UI delta (phosphene)"); ax[0].plot(vt, label="raw VT bytes", alpha=.6)
ax[0].set_yscale("log"); ax[0].set_xlabel("keyframe"); ax[0].set_ylabel("bytes"); ax[0].legend()
ax[1].plot(np.cumsum(full), label="full"); ax[1].plot(np.cumsum(inc), label="delta"); ax[1].plot(np.cumsum(vt), label="VT"); ax[1].legend(); ax[1].set_title("cumulative bytes")
print(f"delta/full = {inc[1:].sum() / max(full[1:].sum(), 1):.2%}   structural updates on {np.mean([s['structural'] for s in stats[1:]]):.0%} of keyframes")

In [ ]:
#@title Interactive replay (terminal | A2UI native | A2UI faithful)
from vttui.reconcile import Reconciler
rec, roles_seq = Reconciler(), []
for f, m in zip(kf, msgs):
    rec.step(f, labeler(f)); roles_seq.append(rec.roles)
replay(kf, roles_seq, msgs)

In [ ]:
#@title Export stream: JSONL + stylesheet (open in packages/viewer/index.html)
from vttui.reconcile import to_jsonl
open(f"{DATA}/stream.a2ui.jsonl", "w").write("".join(to_jsonl(m) for m in msgs))
open(f"{DATA}/stream.css", "w").write(rec.stylesheet())
print(f"{DATA}/stream.a2ui.jsonl")